In [50]:
# ------------------------------------------------------------------------------------------------------------------------
# FANBEATS: Frequency and Attention-augmented Neural Basis Expansion Analysis for interpretable Time Series forecasting
#
# Copyright (c) 2026 Danish Abbas
#
# This file is part of the FANBEATS project.
# Licensed under the Creative Commons Attribution-NonCommercial
# 4.0 International License (CC BY-NC 4.0).
#
# See the LICENSE file in the root directory for full details.
# ------------------------------------------------------------------------------------------------------------------------

# FANBEATS Interpretability analysis notebook

In [51]:
import os
import sys
import random
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent

sys.path.insert(0, str(REPO_ROOT))

print("Repo root added:", REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch as t
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 32 # setting it for reproducibility of results

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
t.manual_seed(SEED)

if t.cuda.is_available():
    t.cuda.manual_seed(SEED)
    t.cuda.manual_seed_all(SEED)

t.backends.cudnn.deterministic = True
t.backends.cudnn.benchmark = False

import matplotlib
from matplotlib.ticker import FuncFormatter

import matplotlib
import matplotlib.font_manager as fm

import matplotlib.pyplot as plt

# according to the journal's requirements
matplotlib.rcParams.update({
    "font.family": "Minion Pro",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 10,
    "axes.edgecolor": "black",
    "axes.linewidth": 1.0,
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": 300,
})

from models.fanbeats_model import FANBEATSModel

from factories import nbeats_factories_modified
from trainers import trainer as model_trainer
from utils.sampler import TimeseriesSampler
from utils.ops import to_tensor, default_device

from configs import config
from utils import solar_wind_dataset as swd

print("Default device:", default_device())

Repo root added: d:\Data Science (Machine Learning)\Time Series Projects Implemented and Developed\FANBEATS
Default device: cuda


In [52]:
OUTPUT_ROOT = REPO_ROOT / config.OUTPUT_ROOT
INTERPRETABILITY_DIR = REPO_ROOT / config.INTERPRETABILITY_DIR
INTERPRETABILITY_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = config.LOOKBACK_LENGTH
HORIZON = config.FORECAST_HORIZON
MODE = config.NBEATS_MODE
USE_ZSCORE = True

EXPERIMENT_TAG = config.make_experiment_tag(MODE, LOOKBACK, HORIZON)

TRAINED_MODELS_DIR = REPO_ROOT / config.TRAINED_MODELS_DIR
MODEL_PATH = TRAINED_MODELS_DIR / f"{EXPERIMENT_TAG}.pt"


In [53]:
# Load data (fixed split, no random splitting)
train_path = str(REPO_ROOT / config.TRAIN_FILE)
val_path = str(REPO_ROOT / config.VAL_FILE)
test_path = str(REPO_ROOT / config.TEST_FILE)

train_arr, val_arr, test_arr = swd.load_train_val_test(
    train_path=train_path,
    val_path=val_path,
    test_path=test_path,
    column="speed",
)

 #merging both the training and validation based on original process of N-BEATS paper
train_full_arr = np.concatenate([train_arr, val_arr], axis=0)

train_dt, val_dt, test_dt = swd.load_train_val_test_datetimes(
    train_path=train_path,
    val_path=val_path,
    test_path=test_path,
    datetime_col="date_time",
    freq="h",
)

train_full_dt = train_dt.append(val_dt)

print("Raw split shapes:")
print("  Train:", train_full_arr.shape)
print("  Test :", test_arr.shape)

Raw split shapes:
  Train: (52608,)
  Test : (8760,)


In [54]:
if USE_ZSCORE:
    mu, sigma = swd.fit_standard_scaler(train_full_arr)
    train_norm = swd.apply_standard_scaler(train_full_arr, mu, sigma)
    test_norm = swd.apply_standard_scaler(test_arr, mu, sigma)
else:
    mu, sigma = 0.0, 1.0
    train_norm = train_full_arr.astype(np.float32)
    test_norm = test_arr.astype(np.float32)

train_list = swd.to_timeseries_list(train_norm)

print(f"Scaler -> mu={mu:.4f}, sigma={sigma:.4f}")
print("Train list lengths:", [len(x) for x in train_list])

Scaler -> mu=419.6638, sigma=90.6219
Train list lengths: [52608]


> Starting to make the Interpretability Visualisations.

In [55]:

# Global cache for all horizons
_INTERPRET_CACHE = {}  # {horizon: {...}}

def interpret_single_horizon(
    model,
    test_windows,
    horizon,
    model_name_tag,
    batch_size,
    mu=None,
    sigma=None,
):


    global _INTERPRET_CACHE


    sample_batch = test_windows[:batch_size]
    x_np = np.stack([item[0] for item in sample_batch], axis=0) 
    y_np = np.stack([item[1] for item in sample_batch], axis=0)

    x_t = to_tensor(x_np).to(default_device())
    x_mask_t = t.ones_like(x_t)

    model.eval()
    with t.no_grad():
        _ = model(x_t, x_mask_t, return_decomposition=True)

    diag = model.get_diagnostics()
    pred_np = diag["forecast"].detach().cpu().numpy() 

    sample_mae = np.mean(np.abs(pred_np - y_np), axis=1)
    idx = np.argsort(sample_mae)[len(sample_mae) // 2]  

    
    def _safe_np(x, idx=None):
        if x is None:
            return None
        if t.is_tensor(x):
            x = x.detach().cpu().numpy()
        if idx is not None and hasattr(x, "__len__") and np.ndim(x) > 0:
            return x[idx]
        return x

    sfam_freq_attn_w = _safe_np(diag.get("freq_attention_weights"), idx=idx)
    mwam_wav_attn_w = _safe_np(diag.get("wav_attention_weights"), idx=idx)

   
    sfam_freqs        = _safe_np(diag.get("sfam_freqs")) 
    sfam_fft_mag      = _safe_np(diag.get("sfam_fft_magnitude"), idx=idx)      
    sfam_band_mask    = _safe_np(diag.get("sfam_band_mask"), idx=idx)          
    sfam_denoise_mask = _safe_np(diag.get("sfam_denoise_mask"), idx=idx)       
    sfam_guided_mask  = _safe_np(diag.get("sfam_freq_guided_mask"), idx=idx)   
    sfam_low_cutoff  = _safe_np(diag.get("sfam_low_cutoff"), idx=idx)
    sfam_high_cutoff = _safe_np(diag.get("sfam_high_cutoff"), idx=idx)

    
    mwam_denoised_coeffs   = _safe_np(diag.get("mwam_denoised_coefficients"), idx=idx)


    cdi_weights        = _safe_np(diag.get("cdi_weights"))    

    trend_norm    = _safe_np(diag.get("trend_forecast"), idx=idx)
    season_norm   = _safe_np(diag.get("seasonality_forecast"), idx=idx)
    forecast_norm = _safe_np(diag.get("forecast"), idx=idx)
    baseline_norm = None
    final_input   = _safe_np(diag.get("final_input"), idx=idx)
    if final_input is not None:
        baseline_norm = final_input[-1]

    if trend_norm is not None:
        H = trend_norm.shape[-1]
        time_idx = np.arange(1, H + 1)
    else:
        time_idx = None

    
    if (mu is not None) and (sigma is not None) and (trend_norm is not None):
        trend_phys    = sigma * trend_norm
        season_phys   = sigma * season_norm
        baseline_phys = mu + sigma * baseline_norm
        forecast_phys = mu + sigma * forecast_norm
    else:
        trend_phys    = None
        season_phys   = None
        baseline_phys = None
        forecast_phys = None


    _INTERPRET_CACHE[horizon] = {
        "attention": {
            "sfam": sfam_freq_attn_w,
            "mwam": mwam_wav_attn_w,
        },
        "sfam_spectral": {
            "freqs":         sfam_freqs,
            "fft_magnitude": sfam_fft_mag,
            "band_mask":     sfam_band_mask,
            "denoise_mask":  sfam_denoise_mask,
            "guided_mask":   sfam_guided_mask,
            "low_cutoff":    sfam_low_cutoff,
            "high_cutoff":   sfam_high_cutoff,
        },
        "mwam_multiscale": {
            "denoised_coefficients": mwam_denoised_coeffs,
        },
        "cdi": {
            "weights": cdi_weights,
        },
        "components": {
            "trend_norm":    trend_norm,
            "season_norm":   season_norm,
            "forecast_norm": forecast_norm,
            "baseline_norm": baseline_norm,
            "trend_phys":    trend_phys,
            "season_phys":   season_phys,
            "forecast_phys": forecast_phys,
            "baseline_phys": baseline_phys,
            "time_idx":      time_idx,
        },
        "model_name_tag": model_name_tag,
    }


In [56]:


def generate_all_interpretability_figures(model_name_tag, output_dir):
    

    global _INTERPRET_CACHE

    output_images_path = Path(output_dir).resolve()
    output_images_path.mkdir(parents=True, exist_ok=True)

    horizons = [24, 48, 72, 96]
    letters_4 = ["(a)", "(b)", "(c)", "(d)"]
    letters_8 = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)", "(g)", "(h)"]


    def _moving_average(x, win=5):
        x = np.asarray(x, dtype=float)
        if win <= 1 or len(x) < win:
            return x
        kernel = np.ones(win) / win
        pad = win // 2
        xpad = np.pad(x, (pad, pad), mode="edge")
        return np.convolve(xpad, kernel, mode="valid")

    def _rowwise_norm(mat):
        mat = np.asarray(mat, dtype=float)
        out = np.zeros_like(mat, dtype=float)
        for i in range(mat.shape[0]):
            row = mat[i]
            rmin, rmax = row.min(), row.max()
            out[i] = (row - rmin) / (rmax - rmin + 1e-12)
        return out

    def _norm01(x):
        x = np.asarray(x, dtype=float)
        xmin, xmax = np.min(x), np.max(x)
        return (x - xmin) / (xmax - xmin + 1e-12)

    def _have(h):
        return h in _INTERPRET_CACHE

    def _get_att(h):
        a = _INTERPRET_CACHE.get(h, {}).get("attention", {})
        sfam = None if a.get("sfam") is None else np.array(a["sfam"], copy=True)
        mwam = None if a.get("mwam") is None else np.array(a["mwam"], copy=True)
        return sfam, mwam

    def _get_sfam_spec(h):
        s = _INTERPRET_CACHE.get(h, {}).get("sfam_spectral", {})
        return {
            "freqs": None if s.get("freqs") is None else np.array(s["freqs"], copy=True),
            "fft_magnitude": None if s.get("fft_magnitude") is None else np.array(s["fft_magnitude"], copy=True),
            "band_mask": None if s.get("band_mask") is None else np.array(s["band_mask"], copy=True),
            "denoise_mask": None if s.get("denoise_mask") is None else np.array(s["denoise_mask"], copy=True),
            "guided_mask": None if s.get("guided_mask") is None else np.array(s["guided_mask"], copy=True),
            "low_cutoff": None if s.get("low_cutoff") is None else np.array(s["low_cutoff"], copy=True),
            "high_cutoff": None if s.get("high_cutoff") is None else np.array(s["high_cutoff"], copy=True),
        }

    def _get_mwam_multi(h):
        m = _INTERPRET_CACHE.get(h, {}).get("mwam_multiscale", {})
        return {
            "denoised_coefficients": None if m.get("denoised_coefficients") is None 
                                    else np.array(m["denoised_coefficients"], copy=True),
        }

        
    def _sparse_ticks(ax, t, num=5):
        """Set ~num evenly spaced ticks for horizon t."""
        if t is None or len(t) == 0:
            return
        t = np.asarray(t)
        #choosing evenly spaced positions
        idx = np.linspace(0, len(t) - 1, num=num, dtype=int)
        ticks = t[idx]
        ax.set_xticks(ticks)
        ax.set_xticklabels([str(int(v)) for v in ticks])


    if all(_have(h) for h in horizons):
        fig_w, fig_h = 7.0, 7.0 * 0.8
        fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h), sharex=False, sharey=False)
        axes = axes.flatten()

        handles_labels = None

        for idx, (ax, H, letter) in enumerate(zip(axes, horizons, letters_4)):
            sfam, mwam = _get_att(H)
            if sfam is None or mwam is None:
                ax.text(0.5, 0.5, f"No attention data (H={H})",
                        ha="center", va="center", fontsize=10)
                ax.set_axis_off()
                continue

            x = np.arange(len(sfam))

            l1, = ax.plot(
                x, sfam,
                label="SFAM Domain",
                color="#1f77b4",
                lw=1.0,
                linestyle="-",
                marker="o",
                markevery=[0, len(x)//3, 2*len(x)//3],
                markersize=4,
            )

            l2, = ax.plot(
                x, mwam,
                label="MWAM Domain",
                color="#ff7f0e",
                lw=1.0,
                linestyle="-",
                marker="s",
                markevery=[len(x)//4, len(x)//2, 3*len(x)//4],
                markersize=4,
            )

            ymin = min(np.min(sfam), np.min(mwam))
            ymax = max(np.max(sfam), np.max(mwam))
            pad = max(1e-6, 0.02 * (ymax - ymin))
            ax.set_ylim(ymin - pad, ymax + pad)

            ticks = np.arange(0, len(x), 50)
            if len(x) and (len(ticks) == 0 or ticks[-1] != len(x)-1):
                ticks = np.append(ticks, len(x)-1)
            ax.set_xticks(ticks)
            if idx < 2:
                ax.set_xticklabels([])
            else:
                ax.set_xticklabels(ticks) 

            ax.tick_params(labelsize=10, labelleft=True)
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.4f}"))

            ax.set_title(f"H = {H} h", fontsize=10)

            ax.text(0.02, 0.98, letter, transform=ax.transAxes,
                    fontsize=10, fontweight="bold", va="top", ha="left",
                    bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

            if handles_labels is None:
                handles_labels = ([l1, l2], ["SFAM Domain", "MWAM Domain"])

        fig.supylabel("Weight", fontsize=10)
        fig.supxlabel("Time [Hour]", fontsize=10)

        if handles_labels is not None:
            handles, labels = handles_labels
            fig.legend(handles, labels, loc="upper center", ncol=2,
                       bbox_to_anchor=(0.5, 1.05), frameon=False, fontsize=10)

        base = output_images_path / f"{model_name_tag}_interpret_Attention_AllHorizons"
        plt.tight_layout(rect=[0, 0, 1, 0.98])
        fig.savefig(base.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
        # fig.savefig(base.with_suffix(".pdf"), dpi=300, bbox_inches="tight", facecolor="white")
        # fig.savefig(base.with_suffix(".eps"), dpi=300, bbox_inches="tight")
        plt.close(fig)

    #

    fig_w, fig_h = 7.0, 7.0 * 1.35
    fig, axes = plt.subplots(4, 2, figsize=(fig_w, fig_h), sharex=False, sharey=False)
    axes = axes.reshape(4, 2)

    for row, H in enumerate(horizons):
        if not _have(H):
            axes[row, 0].text(0.5, 0.5, f"No data (H={H})", ha="center", va="center")
            axes[row, 1].text(0.5, 0.5, f"No data (H={H})", ha="center", va="center")
            continue

        C = _INTERPRET_CACHE[H]["components"]

        t = C["time_idx"]
        forecast = C["forecast_phys"]
        baseline = C["baseline_phys"]
        trend = C["trend_phys"]
        season = C["season_phys"]

        recon = baseline + trend + season

        markevery = [0, len(t)//3, 2*len(t)//3, len(t)-1]

        
        markevery_f = [0, len(t)//2]             
        markevery_r = [len(t)//4, len(t)-1]      


        axes[row, 0].plot(
            t, forecast,
            color="#1f77b4", lw=1.2,
            marker="o", markersize=6,
            markevery=markevery_f,
        )

        axes[row, 0].plot(
            t, recon,
            color="#ff7f0e", lw=1.2,
            marker="s", markersize=4,
            markevery=markevery_r,
        )

        axes[row, 0].set_ylabel(f"H = {H} h", fontsize=10)
        # axes[row, 0].grid(alpha=0.2)

        
        axes[row, 1].plot(
            t, trend,
            color="#ff7f0e", lw=1.2,
            marker="^", markersize=4,
            markevery=markevery,
        )

        axes[row, 1].plot(
            t, season,
            color="#1f77b4", lw=1.2,
            marker="d", markersize=4,
            markevery=markevery,
        )

        # axes[row, 1].grid(alpha=0.2)

   
    axes[0, 0].set_title("Forecast vs Reconstruction", fontsize=10)
    axes[0, 1].set_title("Trend and Seasonality Contributions", fontsize=10)

    fig.supxlabel("Forecast Step [Hour]", fontsize=10)
    fig.supylabel("Velocity [km/s]", fontsize=10)

    
    handles = [
        plt.Line2D([], [], color="#1f77b4", lw=1.2, marker="o"),                    
        plt.Line2D([], [], color="#ff7f0e", lw=1.2, marker="s"),     
        plt.Line2D([], [], color="#ff7f0e", lw=1.2, marker="^"),                     
        plt.Line2D([], [], color="#1f77b4", lw=1.2, marker="d"),                    
    ]

    labels = [
        "Forecast",
        "Reconstruction",
        "Trend contribution",
        "Seasonality contribution",
    ]
    
    fig.legend(handles, labels, loc="upper center", ncol=4,
            bbox_to_anchor=(0.5, 1.02), frameon=False)


    fig.text(
        1.0, 0.5, "Additive contribution [km/s]",
        va="center", ha="center",
        rotation="vertical", fontsize=10
    )
    
    base = output_images_path / f"{model_name_tag}_interpret_DecompositionCombined_AllHorizons"
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(base.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".pdf"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".eps"), dpi=300, bbox_inches="tight")
    plt.close(fig)

   
    fig_w, fig_h = 7.0, 7.0 * 0.9
    fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h), sharex=False, sharey=False)
    axes = axes.flatten()

    handles_labels = None

    for idx, (ax, H, letter) in enumerate(zip(axes, horizons, letters_4)):
        if not _have(H):
            ax.text(0.5, 0.5, f"No SFAM spectral data (H={H})",
                    ha="center", va="center", fontsize=10)
            ax.set_axis_off()
            continue

        S = _get_sfam_spec(H)
        freqs = S["freqs"]
        fft_mag = S["fft_magnitude"]
        band_mask = S["band_mask"]
        denoise_mask = S["denoise_mask"]
        guided_mask = S["guided_mask"]
        low_cutoff = S["low_cutoff"]
        high_cutoff = S["high_cutoff"]

        if freqs is None or fft_mag is None or band_mask is None or denoise_mask is None or guided_mask is None:
            ax.text(0.5, 0.5, f"No SFAM spectral data (H={H})",
                    ha="center", va="center", fontsize=10)
            ax.set_axis_off()
            continue

        pos = freqs > 0
        f = freqs[pos]
        mag = np.asarray(fft_mag[pos], dtype=float)
        band = np.asarray(band_mask[pos], dtype=float)
        denoise = np.asarray(denoise_mask[pos], dtype=float)
        guided = np.asarray(guided_mask[pos], dtype=float)

        period_hours = 1.0 / (f + 1e-12)

        initial_response = band * denoise
        final_response = band * denoise * guided

        mag_plot = mag / (np.max(mag) + 1e-12)
        init_plot = initial_response
        final_plot = final_response

        mag_plot = _moving_average(mag_plot, win=3)
        init_plot = _moving_average(init_plot, win=5)
        final_plot = _moving_average(final_plot, win=5)

        order = np.argsort(period_hours)
        period_plot = period_hours[order]
        mag_plot = mag_plot[order]
        init_plot = init_plot[order]
        final_plot = final_plot[order]

        valid = period_plot <= 96
        period_plot = period_plot[valid]
        mag_plot = mag_plot[valid]
        init_plot = init_plot[valid]
        final_plot = final_plot[valid]

        marker_positions = [12, 36, 60, 84]
        mark_idx = [np.argmin(np.abs(period_plot - p)) for p in marker_positions]

        l1, = ax.plot(
            period_plot, mag_plot,
            lw=1.2,
            color="#4c78a8",
            label="Input Spectrum",
            linestyle="-",
            marker="o",
            markevery=mark_idx,
            markersize=4,
        )

        l2, = ax.plot(
            period_plot, init_plot,
            lw=1.2,
            color="#f58518",
            label="Initial Spectral Response",
            linestyle="-",
            marker="s",
            markevery=mark_idx,
            markersize=4,
        )

        l3, = ax.plot(
            period_plot, final_plot,
            lw=1.2,
            color="#54a24b",
            label="Final Spectral Response",
            linestyle="-",
            marker="^",
            markevery=mark_idx,
            markersize=4,
        )

        ax.set_xlim(period_plot.min(), period_plot.max())
        ax.set_ylim(-0.02, 1.05)
        ax.tick_params(labelsize=10)

        desired_ticks = [6, 12, 24, 48, 72, 96]
        ticks = [tt for tt in desired_ticks if period_plot.min() <= tt <= period_plot.max()]
        if len(ticks) > 0:
            ax.set_xticks(ticks)
            if idx < 2:
                ax.set_xticklabels([]) 
            else:
                ax.set_xticklabels([str(int(tt)) for tt in ticks])

        if low_cutoff is not None:
            lc = float(np.mean(low_cutoff))
            if lc > 0:
                ax.axvline(1.0 / lc, color="k", linestyle="--", lw=0.8, alpha=0.6)
        if high_cutoff is not None:
            hc = float(np.mean(high_cutoff))
            if hc > 0:
                ax.axvline(1.0 / hc, color="k", linestyle="--", lw=0.8, alpha=0.6)

        ax.set_title(f"H = {H} h", fontsize=10)

        ax.text(0.02, 0.98, letter, transform=ax.transAxes,
                fontsize=10, fontweight="bold", va="top", ha="left",
                bbox=dict(facecolor='white', edgecolor='none', pad=1.5))

        if handles_labels is None:
            handles_labels = (
                [l1, l2, l3],
                ["Input Spectrum", "Initial Spectral Response", "Final Spectral Response"],
            )

    fig.supylabel("Normalised Magnitude / Weight", fontsize=10)
    fig.supxlabel("Period [Hour]", fontsize=10)

    if handles_labels is not None:
        handles, labels = handles_labels
        fig.legend(handles, labels, loc="upper center", ncol=3,
                   bbox_to_anchor=(0.5, 1.05), frameon=False, fontsize=10)

    base = output_images_path / f"{model_name_tag}_interpret_SFAM_SpectralResponseRefined_AllHorizons"
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(base.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".pdf"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".eps"), dpi=300, bbox_inches="tight")
    plt.close(fig)

 
    fig_w, fig_h = 7.0, 7.0 * 0.9
    fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h), sharex=False, sharey=True)
    axes = axes.flatten()

    last_im = None

    for idx, (ax, H, letter) in enumerate(zip(axes, horizons, letters_4)):
        if not _have(H):
            ax.text(0.5, 0.5, f"No wavelet coefficients (H={H})",
                    ha="center", va="center", fontsize=10)
            ax.set_axis_off()
            continue

        M = _get_mwam_multi(H)
        denoised_coeffs = M["denoised_coefficients"]

        if denoised_coeffs is None:
            ax.text(0.5, 0.5, f"No wavelet coefficients (H={H})",
                    ha="center", va="center", fontsize=10)
            ax.set_axis_off()
            continue

        coeff_plot = np.abs(denoised_coeffs)
        coeff_plot = _rowwise_norm(coeff_plot)

        last_im = ax.imshow(
            coeff_plot,
            aspect="auto",
            origin="lower",
            cmap="viridis",
            vmin=0.0,
            vmax=1.0,
        )

        ax.set_yticks(np.arange(coeff_plot.shape[0]))
        ax.set_yticklabels(
            ["S1 (finest)", "S2", "S3", "S4 (coarsest)"],
            fontsize=10,
        )

        xt = np.arange(0, coeff_plot.shape[1], 24)
        if len(xt) == 0 or xt[-1] != coeff_plot.shape[1] - 1:
            xt = np.append(xt, coeff_plot.shape[1] - 1)
        ax.set_xticks(xt)
        if idx < 2:
            ax.set_xticklabels([])
        else:
            ax.set_xticklabels([str(int(v)) for v in xt], fontsize=10)

        ax.tick_params(axis='both', labelsize=10)
        ax.set_title(f"H = {H} h", fontsize=10)

        ax.text(
            0.02, 0.98, letter,
            transform=ax.transAxes,
            fontsize=10, fontweight="bold",
            va="top", ha="left",
            color="white",
            bbox=dict(facecolor='none', edgecolor='white', pad=1.5),
        )

    # axes[0].set_ylabel("Scale", fontsize=10)
    fig.supylabel("Scale", fontsize=10)
    fig.supxlabel("Time [Hour]", fontsize=10)

    plt.tight_layout(rect=[0, 0, 0.93, 1])

    if last_im is not None:
        cax = fig.add_axes([0.945, 0.18, 0.018, 0.64])
        cbar = fig.colorbar(last_im, cax=cax)
        cbar.ax.tick_params(labelsize=10)
        cbar.set_label("Row-wise Normalised |Coefficient|", fontsize=10)

    base = output_images_path / f"{model_name_tag}_interpret_MWAM_HeatmapOnly_AllHorizons"
    fig.savefig(base.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".pdf"), dpi=300, bbox_inches="tight", facecolor="white")
    # fig.savefig(base.with_suffix(".eps"), dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("[generate_all_interpretability_figures] All merged figures generated.")
    
    cdi_rows = []
    for H in horizons:
        if not _have(H):
            continue
        cdi = _INTERPRET_CACHE[H].get("cdi", {})
        w = cdi.get("weights")
        if w is None:
            continue
        w = np.asarray(w, dtype=float).flatten()
        if len(w) == 3:
            cdi_rows.append([H, w[0], w[1], w[2]])

    if len(cdi_rows) > 0:
        cdi_rows = sorted(cdi_rows, key=lambda x: x[0])
        cdi_arr = np.array(cdi_rows)
        Hvals = cdi_arr[:, 0]
        fft_att = cdi_arr[:, 1]
        fft_guided = cdi_arr[:, 2]
        wav_att = cdi_arr[:, 3]

        fig, ax = plt.subplots(figsize=(7, 4.0))

        ax.plot(Hvals, fft_att, label="FFT-att", marker="o", lw=1.5)
        ax.plot(Hvals, fft_guided, label="FFT-guided", marker="s", lw=1.5)
        ax.plot(Hvals, wav_att, label="Wavelet-att", marker="^", lw=1.5)

        ax.set_xlabel("Forecast Horizon [Hour]", fontsize=10)
        ax.set_ylabel("CDI Fusion Weight", fontsize=10)
        ax.set_ylim(0, 1)
        # ax.grid(True, alpha=0.3)
        ax.legend(fontsize=10)

        base = output_images_path / f"{model_name_tag}_interpret_CDI_FusionWeights_AllHorizons"
        plt.tight_layout()
        fig.savefig(base.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
        # fig.savefig(base.with_suffix(".pdf"), dpi=300, bbox_inches="tight", facecolor="white")
        # fig.savefig(base.with_suffix(".eps"), dpi=300, bbox_inches="tight")
        plt.close(fig)



In [57]:

_INTERPRET_CACHE = {}

HORIZONS = [24, 48, 72, 96]

for H in HORIZONS:
    EXPERIMENT_TAG = config.make_experiment_tag(MODE, LOOKBACK, H)

    MODEL_PATH = TRAINED_MODELS_DIR / f"{EXPERIMENT_TAG}.pt"
    print(f"\n=== Horizon {H} | Loading model from {MODEL_PATH.name} ===")

    params = config.MODEL_PARAMS["interpretable"]
    base_model = nbeats_factories_modified.interpretable(
        input_size=LOOKBACK,
        output_size=H,
        trend_blocks=params["trend_blocks"],
        trend_layers=params["trend_layers"],
        trend_layer_size=params["trend_layer_size"],
        degree_of_polynomial=params["degree_of_polynomial"],
        seasonality_blocks=params["seasonality_blocks"],
        seasonality_layers=params["seasonality_layers"],
        seasonality_layer_size=params["seasonality_layer_size"],
        num_of_harmonics=params["num_of_harmonics"],
    )

    model = FANBEATSModel(
        nbeats_model=base_model,
        lookback_window=LOOKBACK,
    ).to(default_device())

    state_dict = t.load(MODEL_PATH, map_location=default_device())
    model.load_state_dict(state_dict)
    model.eval()

    test_windows = swd.sliding_windows(
        test_norm,
        insample=LOOKBACK,
        outsample=H,
        step=config.EVAL_WINDOW_STEP,
    )

    interpret_single_horizon(
        model=model,
        test_windows=test_windows,
        horizon=H,
        model_name_tag=EXPERIMENT_TAG,
        batch_size=32,
        mu=mu,
        sigma=sigma,
    )

FINAL_TAG = "fanbeats_interpretable"

generate_all_interpretability_figures(
    model_name_tag=FINAL_TAG,
    output_dir=INTERPRETABILITY_DIR,
)



=== Horizon 24 | Loading model from fanbeats_interpretable_lb96_h24.pt ===

=== Horizon 48 | Loading model from fanbeats_interpretable_lb96_h48.pt ===

=== Horizon 72 | Loading model from fanbeats_interpretable_lb96_h72.pt ===

=== Horizon 96 | Loading model from fanbeats_interpretable_lb96_h96.pt ===
[generate_all_interpretability_figures] All merged figures generated.
